# AerisPlane — Aerodynamic Analysis

This notebook walks through the `aerisplane.aero` module: building an aircraft, running all four solver methods, and visualising the results.

| Method | Key | Physics |
|---|---|---|
| Vortex Lattice | `vlm` | Inviscid, linear. CDi only. |
| Lifting Line | `lifting_line` | Viscous + nonlinear. NeuralFoil section polars at effective alpha. Single solve. |
| Nonlinear Lifting Line | `nonlinear_lifting_line` | Viscous + nonlinear. Fixed-point iteration updating NeuralFoil RHS until vortex strengths converge. Captures stall. |
| AeroBuildup | `aero_buildup` | Semi-empirical. NeuralFoil per section + Jorgensen fuselage. Full CDi + CDp split. |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import aerisplane as ap
from aerisplane.aero import analyze
from aerisplane.aero.result import AeroResult

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

---
## 1 — Build the aircraft

A conventional RC aircraft: tapered main wing + horizontal tail + vertical tail + fuselage.

In [2]:
# ── Airfoils ──────────────────────────────────────────────────────────────
ag35     = ap.Airfoil(name="ag35")          # thin cambered — good at low Re
naca0009 = ap.Airfoil.from_naca("0009")    # symmetric thin — tail surfaces

# ── Main wing ─────────────────────────────────────────────────────────────
# Tapered: 0.28 m root → 0.14 m tip, 1.2 m semispan, −2 deg washout at tip
main_wing = ap.Wing(
    name="main_wing",
    xsecs=[
        ap.WingXSec(xyz_le=[0.00, 0.00, 0.00], chord=0.28, twist= 0.0, airfoil=ag35),
        ap.WingXSec(xyz_le=[0.03, 0.60, 0.02], chord=0.21, twist=-1.0, airfoil=ag35),
        ap.WingXSec(xyz_le=[0.07, 1.20, 0.05], chord=0.14, twist=-2.0, airfoil=ag35),
    ],
    symmetric=True,
)

# ── Horizontal tail ───────────────────────────────────────────────────────
htail = ap.Wing(
    name="htail",
    xsecs=[
        ap.WingXSec(xyz_le=[0.95, 0.00, 0.00], chord=0.10, airfoil=naca0009),
        ap.WingXSec(xyz_le=[0.97, 0.28, 0.00], chord=0.07, airfoil=naca0009),
    ],
    symmetric=True,
)

# ── Vertical tail ─────────────────────────────────────────────────────────
vtail = ap.Wing(
    name="vtail",
    xsecs=[
        ap.WingXSec(xyz_le=[0.92, 0.00, 0.00], chord=0.12, airfoil=naca0009),
        ap.WingXSec(xyz_le=[0.95, 0.00, 0.18], chord=0.08, airfoil=naca0009),
    ],
    symmetric=False,
)

# ── Fuselage ──────────────────────────────────────────────────────────────
fuselage = ap.Fuselage(
    name="fuselage",
    x_le=0.0, y_le=0.0, z_le=0.0,
    xsecs=[
        ap.FuselageXSec(x=0.00, radius=0.010),
        ap.FuselageXSec(x=0.08, radius=0.045),
        ap.FuselageXSec(x=0.25, radius=0.055),
        ap.FuselageXSec(x=0.65, radius=0.045),
        ap.FuselageXSec(x=0.90, radius=0.020),
        ap.FuselageXSec(x=1.05, radius=0.008),
    ],
)

# ── Assemble ──────────────────────────────────────────────────────────────
aircraft = ap.Aircraft(
    name="CoreFly",
    wings=[main_wing, htail, vtail],
    fuselages=[fuselage],
)

mw = aircraft.main_wing()
print(f"Main wing area   : {mw.area():.4f} m²")
print(f"Main wing span   : {mw.span():.4f} m")
print(f"Aspect ratio     : {mw.aspect_ratio():.2f}")
print(f"MAC              : {mw.mean_aerodynamic_chord():.4f} m")
print(f"Taper ratio      : {mw.taper_ratio():.3f}")
print(f"LE sweep         : {mw.sweep_le():.2f} deg")

Main wing area   : 0.5040 m²
Main wing span   : 2.4000 m
Aspect ratio     : 11.43
MAC              : 0.2217 m
Taper ratio      : 0.500
LE sweep         : 3.34 deg


---
## 2 — Single operating point

Cruise: 16 m/s, 300 m altitude, α = 4 deg.

In [3]:
cruise = ap.FlightCondition(velocity=16.0, altitude=300.0, alpha=4.0)

# VLM (inviscid)
r_vlm = analyze(aircraft, cruise, method="vlm")
r_vlm.report()

----------------------------------------------------
  AeroResult — method: vlm
----------------------------------------------------
  Operating point:
    alpha          =     4.0000  deg
    beta           =     0.0000  deg
    velocity       =    16.0000  m/s
    altitude       =      300.0  m
    dyn. pressure  =     152.33  Pa
    Reynolds (MAC) =     237142
----------------------------------------------------
  Reference geometry:
    S_ref = 0.5040 m^2   c_ref = 0.2217 m   b_ref = 2.4000 m
----------------------------------------------------
  Force coefficients (wind axes):
    CL   =   0.594211
    CD   =   0.009559
    CDi  =   0.009559  (induced)
    CY   =  -0.000000
----------------------------------------------------
  Moment coefficients (body axes):
    Cl   =   0.000000  (roll)
    Cm   =  -0.322310  (pitch)
    Cn   =   0.000000  (yaw)
----------------------------------------------------
  Forces (wind axes):
    L    =     45.620  N
    D    =      0.734  N
    Y    

In [4]:
# AeroBuildup (viscous — full CDi + CDp breakdown)
r_ab = analyze(aircraft, cruise, method="aero_buildup", model_size="xxsmall")
r_ab.report()

----------------------------------------------------
  AeroResult — method: aero_buildup
----------------------------------------------------
  Operating point:
    alpha          =     4.0000  deg
    beta           =     0.0000  deg
    velocity       =    16.0000  m/s
    altitude       =      300.0  m
    dyn. pressure  =     152.33  Pa
    Reynolds (MAC) =     237142
----------------------------------------------------
  Reference geometry:
    S_ref = 0.5040 m^2   c_ref = 0.2217 m   b_ref = 2.4000 m
----------------------------------------------------
  Force coefficients (wind axes):
    CL   =   0.749120
    CD   =   0.035279
    CDi  =   0.019140  (induced)
    CDp  =   0.016139  (parasitic)
    CY   =   0.000000
----------------------------------------------------
  Moment coefficients (body axes):
    Cl   =   0.000000  (roll)
    Cm   =  -0.471231  (pitch)
    Cn   =  -0.000000  (yaw)
----------------------------------------------------
  Forces (wind axes):
    L    =     

---
## 3 — Polar sweep — all four methods

Sweep α from −4 to 12 deg and compare CL, CD, and Cm across all solvers.

> **Note on NLL and stall:** NonlinearLiftingLine uses a plain NumPy fixed-point iteration (no CasADi). Each iteration solves the linear LL system with the RHS updated from NeuralFoil at the current effective alpha, then under-relaxes. Convergence is typically 10–30 iterations. Near stall (≈13–14° for this wing at Re~200k) the NeuralFoil polar becomes highly nonlinear and the iteration can become slow, so the polar sweep is capped at 12° — safely pre-stall. VLM and LiftingLine have no such limit because they do not model viscous stall.

In [5]:
ALPHAS   = np.linspace(-4, 12, 9)   # capped at 12° — NLL diverges past stall (~13-14°)
VELOCITY = 16.0
ALTITUDE = 300.0

methods = {
    "vlm":                    dict(method="vlm"),
    "lifting_line":           dict(method="lifting_line",          model_size="xxsmall"),
    "nonlinear_lifting_line": dict(method="nonlinear_lifting_line", spanwise_resolution=3),
    "aero_buildup":           dict(method="aero_buildup",           model_size="xxsmall"),
}

polars = {}   # method → list[AeroResult | None]  (None = solver failed at that point)

import warnings
for label, kwargs in methods.items():
    print(f"Running {label} ...", end=" ", flush=True)
    results = []
    failed  = 0
    for alpha in ALPHAS:
        cond = ap.FlightCondition(velocity=VELOCITY, altitude=ALTITUDE, alpha=float(alpha))
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                results.append(analyze(aircraft, cond, **kwargs))
        except Exception:
            results.append(None)
            failed += 1
    polars[label] = results
    tag = f"  ({failed} point(s) skipped)" if failed else ""
    print(f"done{tag}")

print("\nAll polars computed.")

Running vlm ... done
Running lifting_line ... done
Running nonlinear_lifting_line ... done
Running aero_buildup ... done

All polars computed.


In [6]:
COLORS  = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0"]
MARKERS = ["o", "s", "^", "D"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Aerodynamic Polars — CoreFly", fontsize=13, fontweight="bold")

for (label, results), color, marker in zip(polars.items(), COLORS, MARKERS):
    # Filter out None entries (failed solves)
    valid   = [(r.alpha, r.CL, r.CD, r.Cm) for r in results if r is not None]
    if not valid:
        continue
    alphas, CLs, CDs, Cms = zip(*valid)

    axes[0].plot(alphas, CLs, marker=marker, color=color, label=label, lw=1.8, ms=5)
    axes[1].plot(CDs,    CLs, marker=marker, color=color, label=label, lw=1.8, ms=5)
    axes[2].plot(alphas, Cms, marker=marker, color=color, label=label, lw=1.8, ms=5)

axes[0].set_xlabel("α [deg]"); axes[0].set_ylabel("CL"); axes[0].set_title("Lift curve")
axes[1].set_xlabel("CD");      axes[1].set_ylabel("CL"); axes[1].set_title("Drag polar")
axes[2].set_xlabel("α [deg]"); axes[2].set_ylabel("Cm"); axes[2].set_title("Pitching moment")
axes[2].axhline(0, color="k", lw=0.8, ls="--")

axes[0].legend(framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.savefig("polar_comparison.png", bbox_inches="tight")
plt.show()
print("Saved polar_comparison.png")

Saved polar_comparison.png


/tmp/ipykernel_98375/1797848433.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4 — Drag breakdown (AeroBuildup)

AeroBuildup decomposes total drag into induced (CDi) and parasitic (CDp).

In [7]:
ab_res = polars["aero_buildup"]

alphas = [r.alpha for r in ab_res]
CLs    = [r.CL    for r in ab_res]
CDi    = [r.CDi   for r in ab_res]
CDp    = [r.CDp   for r in ab_res]
CD     = [r.CD    for r in ab_res]
LoD    = [cl / cd if cd > 0 else 0.0 for cl, cd in zip(CLs, CD)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("AeroBuildup — drag breakdown", fontsize=13, fontweight="bold")

# Stacked area — CDi and CDp vs alpha
axes[0].stackplot(alphas, CDp, CDi,
                  labels=["CDp (parasitic)", "CDi (induced)"],
                  colors=["#FF7043", "#42A5F5"], alpha=0.75)
axes[0].plot(alphas, CD, "k-", lw=1.5, label="CD total")
axes[0].set_xlabel("α [deg]")
axes[0].set_ylabel("Drag coefficient")
axes[0].set_title("CDi vs CDp vs alpha")
axes[0].legend(fontsize=9)

# L/D vs alpha
axes[1].plot(alphas, LoD, "o-", color="#4CAF50", lw=2, ms=6)
best_idx = int(np.argmax(LoD))
axes[1].axvline(alphas[best_idx], color="gray", ls="--", lw=1)
axes[1].annotate(
    f"Best L/D = {LoD[best_idx]:.1f}\n@ α = {alphas[best_idx]:.1f}°",
    xy=(alphas[best_idx], LoD[best_idx]),
    xytext=(alphas[best_idx] + 1.8, LoD[best_idx] * 0.92),
    arrowprops=dict(arrowstyle="->", color="gray"),
    fontsize=9,
)
axes[1].set_xlabel("α [deg]")
axes[1].set_ylabel("L / D")
axes[1].set_title("Lift-to-drag ratio")

plt.tight_layout()
plt.savefig("drag_breakdown.png", bbox_inches="tight")
plt.show()
print("Saved drag_breakdown.png")

Saved drag_breakdown.png


/tmp/ipykernel_98375/503736352.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5 — Velocity sweep at fixed alpha

How CL, CD, Reynolds number, and lift force change with airspeed.

In [8]:
VELOCITIES  = np.linspace(8, 28, 12)
ALPHA_FIXED = 4.0

vel_results = [
    analyze(
        aircraft,
        ap.FlightCondition(velocity=float(v), altitude=300.0, alpha=ALPHA_FIXED),
        method="aero_buildup", model_size="xxsmall",
    )
    for v in VELOCITIES
]

vs  = [r.velocity         for r in vel_results]
CLs = [r.CL               for r in vel_results]
CDs = [r.CD               for r in vel_results]
Res = [r.reynolds_number  for r in vel_results]
Ls  = [r.L                for r in vel_results]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f"Velocity sweep — α = {ALPHA_FIXED}°, AeroBuildup", fontsize=13, fontweight="bold")

specs = [
    (axes[0,0], CLs,              "CL",         "CL vs velocity",          "#2196F3"),
    (axes[0,1], CDs,              "CD",         "CD vs velocity",          "#FF5722"),
    (axes[1,0], [r/1e3 for r in Res], "Re × 10³", "Reynolds number",      "#4CAF50"),
    (axes[1,1], Ls,               "Lift [N]",   "Lift force vs velocity",  "#9C27B0"),
]
for ax, vals, ylabel, title, color in specs:
    ax.plot(vs, vals, "o-", color=color, lw=2, ms=6)
    ax.set_xlabel("Velocity [m/s]")
    ax.set_ylabel(ylabel)
    ax.set_title(title)

plt.tight_layout()
plt.savefig("velocity_sweep.png", bbox_inches="tight")
plt.show()
print("Saved velocity_sweep.png")

Saved velocity_sweep.png


/tmp/ipykernel_98375/3042261266.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6 — Altitude sweep

Lower density at altitude → lower Reynolds number → higher CDp (NeuralFoil captures this).

In [9]:
ALTITUDES = [0, 200, 500, 1000, 1500, 2000, 2500, 3000]

alt_results = [
    analyze(
        aircraft,
        ap.FlightCondition(velocity=16.0, altitude=float(alt), alpha=4.0),
        method="aero_buildup", model_size="xxsmall",
    )
    for alt in ALTITUDES
]

alts = [r.altitude        for r in alt_results]
CDps = [r.CDp             for r in alt_results]
CDis = [r.CDi             for r in alt_results]
Res  = [r.reynolds_number for r in alt_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Altitude sweep — V = 16 m/s, α = 4°", fontsize=13, fontweight="bold")

axes[0].plot(alts, CDps, "o-", color="#FF7043", lw=2, ms=6, label="CDp (parasitic)")
axes[0].plot(alts, CDis, "s-", color="#42A5F5", lw=2, ms=6, label="CDi (induced)")
axes[0].set_xlabel("Altitude [m]")
axes[0].set_ylabel("Drag coefficient")
axes[0].set_title("Drag components vs altitude")
axes[0].legend(fontsize=9)

axes[1].plot(alts, [r/1e3 for r in Res], "o-", color="#4CAF50", lw=2, ms=6)
axes[1].set_xlabel("Altitude [m]")
axes[1].set_ylabel("Re × 10³")
axes[1].set_title("Reynolds number vs altitude")

plt.tight_layout()
plt.savefig("altitude_sweep.png", bbox_inches="tight")
plt.show()
print("Saved altitude_sweep.png")

Saved altitude_sweep.png


/tmp/ipykernel_98375/3266648756.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7 — Sideslip sweep (β)

VLM handles asymmetric conditions well — no section polar required.

In [10]:
BETAS = np.linspace(-10, 10, 9)

beta_results = [
    analyze(
        aircraft,
        ap.FlightCondition(velocity=16.0, altitude=300.0, alpha=4.0, beta=float(b)),
        method="vlm",
    )
    for b in BETAS
]

betas = [r.beta for r in beta_results]
CYs   = [r.CY   for r in beta_results]
Cls   = [r.Cl   for r in beta_results]
Cns   = [r.Cn   for r in beta_results]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Sideslip sweep — VLM, α = 4°", fontsize=13, fontweight="bold")

specs = [
    (axes[0], CYs, "CY (side force)", "#2196F3"),
    (axes[1], Cls, "Cl (roll)",       "#FF5722"),
    (axes[2], Cns, "Cn (yaw)",        "#9C27B0"),
]
for ax, vals, ylabel, color in specs:
    ax.plot(betas, vals, "o-", color=color, lw=2, ms=6)
    ax.axhline(0, color="k", lw=0.7, ls="--")
    ax.axvline(0, color="k", lw=0.7, ls="--")
    ax.set_xlabel("β [deg]")
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)

plt.tight_layout()
plt.savefig("sideslip_sweep.png", bbox_inches="tight")
plt.show()
print("Saved sideslip_sweep.png")

Saved sideslip_sweep.png


/tmp/ipykernel_98375/3227172544.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 8 — Solver comparison at cruise

Bar chart comparing CL, CD, and Cm from all four solvers at the same operating point.

In [11]:
# Find the polar index closest to alpha = 4 deg and reuse computed data
cruise_alpha_idx = int(np.argmin(np.abs(ALPHAS - 4.0)))

# Only include methods that converged at the cruise alpha
cruise_results = {
    label: polars[label][cruise_alpha_idx]
    for label in methods
    if polars[label][cruise_alpha_idx] is not None
}

labels    = list(cruise_results.keys())
short_lbl = {"vlm": "VLM", "lifting_line": "LL",
             "nonlinear_lifting_line": "NLL", "aero_buildup": "AeroBuildup"}
CLvals = [cruise_results[m].CL for m in labels]
CDvals = [cruise_results[m].CD for m in labels]
Cmvals = [cruise_results[m].Cm for m in labels]

x = np.arange(len(labels))

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
fig.suptitle("Solver comparison — cruise (α ≈ 4°, V = 16 m/s)", fontsize=13, fontweight="bold")

for ax, vals, title, color in zip(
    axes,
    [CLvals, CDvals, Cmvals],
    ["CL", "CD", "Cm"],
    ["#2196F3", "#FF5722", "#4CAF50"],
):
    bars = ax.bar(x, vals, color=color, alpha=0.75, edgecolor="k", lw=0.6)
    ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([short_lbl[m] for m in labels], fontsize=9)
    ax.set_title(title)
    ax.set_ylabel(title)

plt.tight_layout()
plt.savefig("solver_comparison.png", bbox_inches="tight")
plt.show()
print("Saved solver_comparison.png")

Saved solver_comparison.png


/tmp/ipykernel_98375/3087731564.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 9 — CLα slope summary

Fit a line through the polar (linear region) to extract CLα [per deg] and CL0 for each solver.

In [12]:
# Reuse the polar data — no extra solver calls needed
print(f"{'Method':<28} {'CLα [/deg]':>12} {'CL0':>10} {'α_trim [deg]':>14}")
print("-" * 68)

cla_table = {}
for label, results in polars.items():
    valid = [(r.alpha, r.CL) for r in results if r is not None]
    if len(valid) < 2:
        print(f"{label:<28} {'(insufficient data)':>38}")
        continue
    als = np.array([v[0] for v in valid])
    cls = np.array([v[1] for v in valid])
    slope, intercept = np.polyfit(als, cls, 1)
    alpha_trim = -intercept / slope if slope != 0 else float("nan")
    cla_table[label] = dict(CLa=slope, CL0=intercept, alpha_trim=alpha_trim)
    print(f"{label:<28} {slope:>12.5f} {intercept:>10.5f} {alpha_trim:>14.2f}")

Method                         CLα [/deg]        CL0   α_trim [deg]
--------------------------------------------------------------------
vlm                               0.09748    0.20712          -2.12
lifting_line                      0.08048    0.26097          -3.24
nonlinear_lifting_line            0.07658    0.20612          -2.69
aero_buildup                      0.08982    0.33059          -3.68


In [13]:
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle("CL–α linear fits by solver", fontsize=13, fontweight="bold")

alpha_fine = np.linspace(ALPHAS[0], ALPHAS[-1], 80)

for (label, results), color, marker in zip(polars.items(), COLORS, MARKERS):
    valid = [(r.alpha, r.CL) for r in results if r is not None]
    if not valid or label not in cla_table:
        continue
    als, cls = zip(*valid)
    cla = cla_table[label]
    fit_line = cla["CLa"] * alpha_fine + cla["CL0"]

    ax.scatter(als, cls, color=color, marker=marker, s=40, zorder=5)
    ax.plot(alpha_fine, fit_line, color=color, lw=1.5, ls="--",
            label=f"{label}  (CLα = {cla['CLa']:.4f} /deg)")

ax.axhline(0, color="k", lw=0.7)
ax.set_xlabel("α [deg]")
ax.set_ylabel("CL")
ax.legend(fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.savefig("cla_fits.png", bbox_inches="tight")
plt.show()
print("Saved cla_fits.png")

Saved cla_fits.png


/tmp/ipykernel_98375/945506497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 10 — Aircraft geometry

AeroSandbox builds a full panel mesh internally. We can visualise the geometry directly from the translated ASB airplane object — both a 3-D wireframe and the classic three-view drawing.

In [14]:
from aerisplane.aero.aerosandbox_backend import aircraft_to_asb
import aerosandbox as asb

asb_plane = aircraft_to_asb(aircraft)

# ── 3-D wireframe ─────────────────────────────────────────────────────────
ax = asb_plane.draw_wireframe(
    show=False,
    use_preset_view_angle="right_isometric",
    set_background_pane_color="whitesmoke",
    set_background_pane_alpha=0.6,
)
ax.set_title("CoreFly — 3-D wireframe", fontsize=12, fontweight="bold")
fig = ax.figure
fig.savefig("geometry_wireframe.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved geometry_wireframe.png")

Saved geometry_wireframe.png


/tmp/ipykernel_98375/2131641004.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# ── Three-view drawing (top / front / side / isometric) ───────────────────
axs_tv = asb_plane.draw_three_view(show=True)
fig_tv  = axs_tv.flat[0].figure
fig_tv.suptitle("CoreFly — three-view", fontsize=12, fontweight="bold", y=1.01)
fig_tv.savefig("geometry_three_view.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved geometry_three_view.png")

/home/stepan/projects/AerisPlane/.venv/lib/python3.12/site-packages/aerosandbox/tools/pretty_plots/formatting.py:410: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved geometry_three_view.png


/tmp/ipykernel_98375/1052907743.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 11 — Pressure distribution (VLM)

VLM solves for a vortex strength on each panel. The net normal force per panel gives the local pressure coefficient:

$$C_p = \frac{F_\text{normal}}{q_\infty \cdot A_\text{panel}}$$

We run VLM at higher resolution (16 spanwise × 6 chordwise) for a smooth distribution, then plot:
- **Top-down view** — Cp coloured across the planform
- **3-D coloured surface** — same data draped over the panel mesh
- **Spanwise CL distribution** — strip-integrated lift coefficient vs spanwise position

In [16]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from aerisplane.aero.aerosandbox_backend import _condition_to_asb

# Run VLM at finer resolution for pressure plots
op_cruise  = _condition_to_asb(ap.FlightCondition(velocity=16.0, altitude=300.0, alpha=4.0))
vlm_hi     = asb.VortexLatticeMethod(
    asb_plane, op_cruise,
    spanwise_resolution=16,
    chordwise_resolution=6,
)
_ = vlm_hi.run()

# ── Extract panel data ─────────────────────────────────────────────────────
centers  = np.array(vlm_hi.collocation_points)   # (N, 3)
areas    = np.array(vlm_hi.areas)                 # (N,)
forces_g = np.array(vlm_hi.forces_geometry)       # (N, 3)
normals  = np.array(vlm_hi.normal_directions)     # (N, 3)

fl = np.array(vlm_hi.front_left_vertices)
bl = np.array(vlm_hi.back_left_vertices)
br = np.array(vlm_hi.back_right_vertices)
fr = np.array(vlm_hi.front_right_vertices)

q_inf = op_cruise.dynamic_pressure()
f_normal = np.einsum('ij,ij->i', forces_g, normals)
Cp = f_normal / (q_inf * areas)

print(f"Panels: {len(Cp)}   Cp min={Cp.min():.4f}  Cp max={Cp.max():.4f}")

Panels: 672   Cp min=-0.0000  Cp max=1.6009


In [17]:
# ── Plot 1: top-down planform Cp map ──────────────────────────────────────
from matplotlib.collections import PolyCollection

cp_max  = np.percentile(Cp, 99)   # clip outliers at LE
cp_min  = 0.0
cmap_cp = plt.cm.RdYlBu_r
norm_cp = plt.Normalize(vmin=cp_min, vmax=cp_max)

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#1a1a2e")

# Build 2-D (x, y) quad patches for each panel — top-down view
quads_2d = [
    [fl[i, [0,1]], bl[i, [0,1]], br[i, [0,1]], fr[i, [0,1]]]
    for i in range(len(Cp))
]
face_colors = [cmap_cp(norm_cp(cp)) for cp in Cp]

pc2d = PolyCollection(quads_2d, facecolors=face_colors,
                       edgecolors="none", linewidth=0)
ax.add_collection(pc2d)
ax.autoscale_view()
ax.set_aspect("equal")
ax.invert_xaxis()    # nose to the right (conventional layout)
ax.set_xlabel("x [m]", color="white")
ax.set_ylabel("y [m]", color="white")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_edgecolor("gray")

sm = plt.cm.ScalarMappable(cmap=cmap_cp, norm=norm_cp)
sm.set_array([])
cb = plt.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
cb.set_label("Cp (net normal)", color="white")
cb.ax.yaxis.set_tick_params(color="white")
plt.setp(cb.ax.yaxis.get_ticklabels(), color="white")

ax.set_title(f"Pressure distribution — top-down view  (α = 4°, V = 16 m/s)",
             color="white", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig("pressure_topdown.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved pressure_topdown.png")

Saved pressure_topdown.png


/tmp/ipykernel_98375/817106444.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# ── Plot 2: 3-D coloured surface ──────────────────────────────────────────
fig3d = plt.figure(figsize=(13, 7))
fig3d.patch.set_facecolor("#1a1a2e")
ax3d  = fig3d.add_subplot(111, projection="3d")
ax3d.set_facecolor("#1a1a2e")

quads_3d = [[fl[i], bl[i], br[i], fr[i]] for i in range(len(Cp))]
face_colors_3d = [cmap_cp(norm_cp(cp)) for cp in Cp]

pc3d = Poly3DCollection(quads_3d, facecolors=face_colors_3d,
                         edgecolors="none", linewidth=0, zsort="average")
ax3d.add_collection3d(pc3d)

# Set axis limits from panel centres
ax3d.set_xlim(centers[:,0].min() - 0.05, centers[:,0].max() + 0.05)
ax3d.set_ylim(centers[:,1].min() - 0.1,  centers[:,1].max() + 0.1)
ax3d.set_zlim(centers[:,2].min() - 0.05, centers[:,2].max() + 0.05)

ax3d.set_xlabel("x [m]", color="white", labelpad=6)
ax3d.set_ylabel("y [m]", color="white", labelpad=6)
ax3d.set_zlabel("z [m]", color="white", labelpad=6)
ax3d.tick_params(colors="white", labelsize=7)
for pane in [ax3d.xaxis.pane, ax3d.yaxis.pane, ax3d.zaxis.pane]:
    pane.set_facecolor("#1a1a2e")
    pane.set_edgecolor("gray")

ax3d.view_init(elev=28, azim=-110)

sm3 = plt.cm.ScalarMappable(cmap=cmap_cp, norm=norm_cp)
sm3.set_array([])
cb3 = fig3d.colorbar(sm3, ax=ax3d, shrink=0.55, pad=0.08)
cb3.set_label("Cp", color="white")
cb3.ax.yaxis.set_tick_params(color="white")
plt.setp(cb3.ax.yaxis.get_ticklabels(), color="white")

ax3d.set_title("Pressure distribution — 3-D view  (α = 4°, V = 16 m/s)",
               color="white", fontsize=12, fontweight="bold", pad=12)

fig3d.tight_layout()
fig3d.savefig("pressure_3d.png", dpi=150, bbox_inches="tight",
              facecolor=fig3d.get_facecolor())
plt.show()
print("Saved pressure_3d.png")

Saved pressure_3d.png


/tmp/ipykernel_98375/2707749382.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# ── Plot 3: spanwise CL distribution ──────────────────────────────────────
# Identify main wing panels (|y| <= semispan + small margin, |x| <= ~0.4)
mw       = aircraft.main_wing()
semispan = mw.semispan()
s_ref    = aircraft.reference_area()
c_ref    = aircraft.reference_chord()

# For each spanwise strip: bin panels by y, sum (Cp * area), divide by local chord
# We use the panel centroid y as the spanwise coordinate
y_c   = centers[:, 1]
x_c   = centers[:, 0]

# Keep only main wing panels (rough bounds)
mw_mask = (np.abs(y_c) <= semispan * 1.05) & (x_c <= 0.35)
y_mw  = y_c[mw_mask]
Cp_mw = Cp[mw_mask]
A_mw  = areas[mw_mask]

# Bin by y-station
n_bins   = 32
y_edges  = np.linspace(0, semispan, n_bins + 1)
y_mid    = 0.5 * (y_edges[:-1] + y_edges[1:])
cl_strip = np.zeros(n_bins)

for k in range(n_bins):
    mask = (np.abs(y_mw) >= y_edges[k]) & (np.abs(y_mw) < y_edges[k+1])
    if mask.sum() == 0:
        continue
    # Local chord at this y by interpolating wing geometry
    y_secs = np.array([xs.xyz_le[1] for xs in mw.xsecs])
    c_secs = np.array([xs.chord      for xs in mw.xsecs])
    c_local = np.interp(y_mid[k], y_secs, c_secs)
    cl_strip[k] = np.sum(Cp_mw[mask] * A_mw[mask]) / (q_inf * c_local * np.diff(y_edges)[k])

# Elliptic reference distribution (same total lift)
CL_total = r_vlm.CL
A_R      = mw.aspect_ratio()
cl_elliptic = (4 * CL_total / (np.pi * A_R)) * np.sqrt(np.maximum(0, 1 - (y_mid / semispan)**2))

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Spanwise lift distribution — VLM (α = 4°, V = 16 m/s)",
             fontsize=12, fontweight="bold")

ax.bar(y_mid, cl_strip, width=np.diff(y_edges)[0] * 0.85,
       color="#2196F3", alpha=0.75, label="VLM strips")
ax.plot(y_mid, cl_elliptic, "r--", lw=2, label="Elliptic reference")

ax.set_xlabel("Spanwise position y [m]")
ax.set_ylabel("Local cl")
ax.set_xlim(0, semispan)
ax.set_ylim(bottom=0)
ax.legend(fontsize=10)
ax.set_title("")

plt.tight_layout()
fig.savefig("spanwise_cl.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved spanwise_cl.png")

Saved spanwise_cl.png


/tmp/ipykernel_98375/745309775.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 12 — Draw geometry as ASB

In [20]:
ap.aero.plot_geometry(style="original",aircraft=aircraft)

/home/stepan/projects/AerisPlane/src/aerisplane/aero/aerosandbox_backend.py:455: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
